# Can SAEs learn from token embeddings?

If an SAE only sees frozen feature combinations in embeddings, can it still learn the underlying true feautres? We test this in a toy model with 50 true features and 25 "tokens embeddings". We create these token embeddings by sampling 25 activations and freezing them. The SAE is then trained just from these 25 frozen activations.

## Install SAELens (you can just run this, may require restarting Colab)

In [1]:
!pip install sae-lens -q

## Training / plotting helpers (you can just run this)

In [4]:
import torch
from typing import Any
from tqdm.auto import tqdm
from typing import Callable, NamedTuple
from scipy.stats import norm
from torch.distributions import MultivariateNormal
import random

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transformer_lens.hook_points import HookedRootModule

from sae_lens.training.sae_trainer import SAETrainer
from sae_lens.config import SAETrainerConfig, LoggingConfig
from sae_lens import TrainingSAE


DEFAULT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Toy Model ####

def orthogonalize(
    num_vectors: int, vector_len: int, target_cos_sim: float = 0
) -> torch.Tensor:
    """Try to make the embeddings have cos sim as close to target as possible."""
    embeddings = torch.randn(num_vectors, vector_len)
    embeddings /= embeddings.norm(p=2, dim=1, keepdim=True)
    embeddings.requires_grad_(True)
    num_vectors = embeddings.shape[0]

    optimizer = torch.optim.Adam([embeddings], lr=0.01)  # type: ignore
    num_steps = 1000

    losses = []

    pbar = tqdm(range(num_steps))
    for step_num in pbar:
        optimizer.zero_grad()

        dot_products = embeddings @ embeddings.T
        diff = dot_products - target_cos_sim
        diff.fill_diagonal_(0)
        loss = diff.pow(2).sum()
        loss += num_vectors * (dot_products.diag() - 1).pow(2).sum()

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        pbar.set_description(f"loss: {loss.item():.3f}")

    embeddings = (
        (embeddings / embeddings.norm(p=2, dim=1, keepdim=True)).detach().clone()
    )
    embeddings.requires_grad_(False)
    return embeddings.detach().clone()


class ToyModel(HookedRootModule):
    def __init__(
        self,
        num_feats: int,
        hidden_dim: int,
        target_cos_sim: float = 0,
        bias: bool = False,
    ):
        super().__init__()
        self.embed = torch.nn.Linear(num_feats, hidden_dim, bias=bias)
        embeddings = orthogonalize(num_feats, hidden_dim, target_cos_sim=target_cos_sim)
        self.embed.weight.data = embeddings.T
        self.setup()

    def forward(self, x: torch.Tensor, **kwargs: Any):  # noqa: ARG002
        return self.embed(x)


#### Generate Batch ####

def get_correlated_features(
    batch_size: int,
    firing_probabilities: torch.Tensor,
    correlation_matrix: torch.Tensor,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """
    Generate correlated binary features using multivariate Gaussian sampling with thresholds.

    Args:
        batch_size: Number of samples to generate
        firing_probabilities: Marginal probabilities for each feature (shape: [num_features])
        correlation_matrix: Correlation matrix between features (shape: [num_features, num_features])
        device: Device to generate samples on

    Returns:
        Binary feature matrix of shape [batch_size, num_features]
    """
    num_features = firing_probabilities.shape[0]

    thresholds = torch.tensor(
        [norm.ppf(1 - p.item()) for p in firing_probabilities], device=device
    )

    mvn = MultivariateNormal(
        loc=torch.zeros(num_features, device=device),
        covariance_matrix=correlation_matrix.to(device),
    )

    gaussian_samples = mvn.sample((batch_size,))  # [batch_size, num_features]
    binary_features = (gaussian_samples > thresholds.unsqueeze(0)).float()

    return binary_features


def get_training_batch(
    batch_size: int,
    firing_probabilities: torch.Tensor,
    std_firing_magnitudes: (
        torch.Tensor | None
    ) = None,
    mean_firing_magnitudes: (
        torch.Tensor | None
    ) = None,
    device: torch.device = DEFAULT_DEVICE,
    modify_firing_features: Callable[[torch.Tensor], torch.Tensor] | None = None,
    correlation_matrix: torch.Tensor | None = None,
):
    if correlation_matrix is not None:
        firing_features = get_correlated_features(
            batch_size, firing_probabilities, correlation_matrix, device
        )
    else:
        firing_features = torch.bernoulli(
            firing_probabilities.unsqueeze(0).expand(batch_size, -1).to(device)
        )

    if std_firing_magnitudes is None:
        std_firing_magnitudes = torch.zeros_like(firing_probabilities)
    if mean_firing_magnitudes is None:
        mean_firing_magnitudes = torch.ones_like(firing_probabilities)
    mean_firing_magnitudes = mean_firing_magnitudes.to(device)
    if modify_firing_features is not None:
        firing_features = modify_firing_features(firing_features)
    firing_features = firing_features.to(device)
    firing_magnitude_delta = torch.normal(
        torch.zeros_like(firing_probabilities)
        .unsqueeze(0)
        .expand(batch_size, -1)
        .to(device),
        std_firing_magnitudes.unsqueeze(0).expand(batch_size, -1).to(device),
    )
    firing_magnitude_delta[firing_features == 0] = 0
    return (firing_features * mean_firing_magnitudes + firing_magnitude_delta).relu()


def generate_correlation_matrix(
    num_features: int,
    positive_ratio: float = 0.5,
    correlation_strength_range: tuple[float, float] = (0.3, 0.8),
    sparsity: float = 0.3,
    seed: int | None = None,
) -> torch.Tensor:
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    matrix = torch.eye(num_features)
    pairs = [(i, j) for i in range(num_features) for j in range(i + 1, num_features)]
    total_pairs = len(pairs)

    if total_pairs == 0:
        return matrix

    num_sparse = int(total_pairs * sparsity)
    num_correlated = total_pairs - num_sparse

    if num_correlated == 0:
        return matrix

    correlated_pairs = random.sample(pairs, num_correlated)
    num_positive = int(num_correlated * positive_ratio)
    min_strength, max_strength = correlation_strength_range

    for i, (pair_i, pair_j) in enumerate(correlated_pairs):
        sign = 1 if i < num_positive else -1
        strength = random.uniform(min_strength, max_strength)
        correlation = sign * strength
        matrix[pair_i, pair_j] = correlation
        matrix[pair_j, pair_i] = correlation

    eigenvals, eigenvecs = torch.linalg.eigh(matrix)
    eigenvals = torch.clamp(eigenvals, min=1e-6)
    matrix = eigenvecs @ torch.diag(eigenvals) @ eigenvecs.T

    diag_sqrt = torch.sqrt(torch.diag(matrix))
    matrix = matrix / (diag_sqrt.unsqueeze(0) * diag_sqrt.unsqueeze(1))
    matrix.fill_diagonal_(1.0)

    return matrix


#### Utils ####

def cos_sims(mat1: torch.Tensor, mat2: torch.Tensor):
    """
    Calculate the cosine similarity between each row of mat1 and each row of mat2.

    Args:
        mat1: A tensor of shape (n_rows1, n_cols1).
        mat2: A tensor of shape (n_rows2, n_cols2).

    Returns:
        A tensor of shape (n_rows1, n_rows2) containing the cosine similarity between each row of mat1 and each row of mat2.
    """
    mat1_normed = mat1 / (mat1.norm(dim=0, keepdim=True))
    mat2_normed = mat2 / (mat2.norm(dim=0, keepdim=True))
    return mat1_normed.T @ mat2_normed


#### Plotting ####

def plot_sae_feat_cos_sims(
    sae: TrainingSAE,
    model: ToyModel,
    title_suffix: str,
    height: int = 400,
    width: int = 800,
    dtick: int = 1,
    show_values: bool = False,
    reorder_features: bool = False,
):
    dec_cos_sims = (
        torch.round(cos_sims(sae.W_dec.T, model.embed.weight) * 100) / 100 + 0.0
    )
    enc_cos_sims = (
        torch.round(cos_sims(sae.W_enc, model.embed.weight) * 100) / 100 + 0.0
    )

    if reorder_features:
        best_feature_matches = torch.argmax(torch.abs(dec_cos_sims), dim=1)
        sorted_indices = torch.argsort(best_feature_matches)
        dec_cos_sims = dec_cos_sims[sorted_indices]
        enc_cos_sims = enc_cos_sims[sorted_indices]

    fig = make_subplots(rows=1, cols=2, subplot_titles=("SAE encoder", "SAE decoder"))
    hovertemplate = "True feature: %{x}<br>SAE Latent: %{y}<br>Cosine Similarity: %{z:.3f}<extra></extra>"

    encoder_args = {
        "z": enc_cos_sims.detach().cpu().numpy(),
        "zmin": -1,
        "zmax": 1,
        "colorscale": "RdBu",
        "showscale": False,
        "hovertemplate": hovertemplate,
    }
    if show_values:
        encoder_args["texttemplate"] = "%{z:.2f}"
        encoder_args["textfont"] = {"size": 10}

    fig.add_trace(go.Heatmap(**encoder_args), row=1, col=1)

    decoder_args = {
        "z": dec_cos_sims.detach().cpu().numpy(),
        "zmin": -1,
        "zmax": 1,
        "colorscale": "RdBu",
        "colorbar": dict(title="cos sim", x=1.0, dtick=1, tickvals=[-1, 0, 1]),
        "hovertemplate": hovertemplate,
    }
    if show_values:
        decoder_args["texttemplate"] = "%{z:.2f}"
        decoder_args["textfont"] = {"size": 10}

    fig.add_trace(go.Heatmap(**decoder_args), row=1, col=2)

    fig.update_layout(
        height=height,
        width=width,
        title_text=f"Cosine Similarity with True Features ({title_suffix})",
    )
    fig.update_xaxes(title_text="True feature", row=1, col=1, dtick=dtick)
    fig.update_xaxes(title_text="True feature", row=1, col=2, dtick=dtick)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=1, dtick=dtick)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=2, dtick=dtick)

    fig.show()


#### Training ####

class DataIterator:
    def __init__(self, model, generate_batch_fn, batch_size: int):
        self.model = model
        self.batch_size = batch_size
        self.generate_batch_fn = generate_batch_fn

    @torch.no_grad()
    def next_batch(self):
        return self.model(self.generate_batch_fn(self.batch_size))

    def __iter__(self):
        return self

    def __next__(self):
        return self.next_batch()


def _save_checkpoint_null(
    trainer: SAETrainer,
    checkpoint_name: int | str,
    wandb_aliases: list[str] | None = None,
):
    pass


def _checkpoint_wrapper(
    checkpoint_fn: Callable[[SAETrainer, int | str], None] | None,
):
    if checkpoint_fn is None:
        return _save_checkpoint_null

    def save_checkpoint(
        trainer: SAETrainer,
        checkpoint_name: int | str,
        wandb_aliases: list[str] | None = None,  # noqa: ARG001
    ):
        checkpoint_fn(trainer, checkpoint_name)

    return save_checkpoint


def train_toy_sae(
    sae: TrainingSAE,
    toy_model: ToyModel,
    activations_batch_provider: Callable[[int], torch.Tensor],
    lr: float = 3e-4,
    training_tokens: int = 15_000_000,
    lr_warm_up_steps: int = 0,
    lr_decay_steps: int = 0,
    device: torch.device = DEFAULT_DEVICE,
    train_batch_size_tokens: int = 1024,
    adam_beta1: float = 0.9,
    adam_beta2: float = 0.999,
) -> None:
    tqdm._instances.clear()  # type: ignore
    training_cfg = SAETrainerConfig(
        n_checkpoints=0,
        checkpoint_path="/home/content/tmp",
        total_training_samples=training_tokens,
        device=str(device),
        autocast=False,
        lr=lr,
        lr_end=lr,
        lr_scheduler_name="constant",
        lr_warm_up_steps=0,
        adam_beta1=0.9,
        adam_beta2=0.999,
        lr_decay_steps=0,
        n_restart_cycles=1,
        train_batch_size_samples=train_batch_size_tokens,
        dead_feature_window=1000,
        feature_sampling_window=2000,
        logger=LoggingConfig(log_to_wandb=False),
        #save_final_checkpoint=False,
    )
    toy_model.eval()
    data_iterator = DataIterator(
        toy_model, activations_batch_provider, train_batch_size_tokens
    )
    trainer = SAETrainer(
        data_provider=data_iterator,
        sae=sae,
        cfg=training_cfg,
    )
    trainer.fit()


## Our toy model

We set up a toy model consisting of 50 mutually orthogonal true features $F = \{f_0, \dots, f_{49}\} \in \mathbb{R}^{100}$. Each feature $f_i$ has a firing probability $P_i = 11 / 50$ (so we get 11 features firing on average). Each feature fires with magnitude $\sim N(1.0, 0.15)$. We create synthetic training inputs for the SAE by sampling true features, and summing all firing features to create a training input. Then, we freeze these as "token embeddings" and use those frozen embeddings to train the SAE.

We construct this toy model below.

In [5]:
from functools import partial
import torch
import pandas as pd

feat_probs = torch.ones(50) * 11 / 50

indices = torch.arange(50) + 1
df = pd.DataFrame({
    "P_i": feat_probs,
    "feature": map(str, indices.tolist()),
})

print("[info] initializing toy_model | num_feats=50 | hidden_dim=100 | target_cos_sim=0.0")
toy_model = ToyModel(num_feats=50, hidden_dim=100).to(DEFAULT_DEVICE)

token_feats = get_training_batch(
    25,
    firing_probabilities=feat_probs,
    std_firing_magnitudes=torch.ones_like(feat_probs) * 0.15,
)
avg_nnz = (token_feats > 0).float().sum(dim=1).mean().item()
print(f"[data] generated 25 token_embeddings | firing_prob={feat_probs[0]:.2f} | avg_nnz={avg_nnz:.1f} | firing_mag=N(1.0, 0.15)")

def generate_batch(num_samples: int):
    random_indices = torch.randint(0, 25, (num_samples,))
    return token_feats[random_indices]


[info] initializing toy_model | num_feats=50 | hidden_dim=100 | target_cos_sim=0.0


[info] initializing toy_model | num_feats=50 | hidden_dim=100 | target_cos_sim=0.0


  0%|          | 0/1000 [00:00<?, ?it/s]

[info] initializing toy_model | num_feats=50 | hidden_dim=100 | target_cos_sim=0.0


  0%|          | 0/1000 [00:00<?, ?it/s]

[data] generated 25 token_embeddings | firing_prob=0.22 | avg_nnz=10.1 | firing_mag=N(1.0, 0.15)


## Training an SAE with 50 latents (same as number of true features)

In [6]:
from sae_lens import StandardTrainingSAE, StandardTrainingSAEConfig

cfg = StandardTrainingSAEConfig(
    l1_coefficient=1.0,
    l1_warm_up_steps=5_000,
    d_in=toy_model.embed.weight.shape[0],
    d_sae=50,
)
sae_50 = StandardTrainingSAE(cfg)
print("[info] training sae | d_sae=50 | l1_coefficient=1.0 | l1_warm_up_steps=5000")
train_toy_sae(sae_50, toy_model, generate_batch)


[info] training sae | d_sae=50 | l1_coefficient=1.0 | l1_warm_up_steps=5000


Training SAE:   0%|          | 0/15000000 [00:00<?, ?it/s]

Let's see if the SAE learned the true features

In [7]:
plot_sae_feat_cos_sims(sae_50, toy_model, "50-latent SAE", dtick=5, reorder_features=True)

In [8]:
sims = cos_sims(sae_50.W_dec.T, toy_model(token_feats).T)
fig = px.imshow(
    sims.detach().cpu().numpy(),
    color_continuous_scale="rdbu",
    zmin=-1,
    zmax=1,
    title="cos sim between SAE decoder and raw token embeddings (50-latent SAE)",
)
fig.show()


It looks like the SAE definitely did not learn the underlying true features, but is a lot closer to learning the token embeddings

## Training an SAE with 25 latents (same as number of tokens)

In [9]:
from sae_lens import StandardTrainingSAE, StandardTrainingSAEConfig

cfg = StandardTrainingSAEConfig(
    l1_coefficient=1.0,
    l1_warm_up_steps=5_000,
    d_in=toy_model.embed.weight.shape[0],
    d_sae=25,
)
sae_25 = StandardTrainingSAE(cfg)
print("[info] training sae | d_sae=25 | l1_coefficient=1.0 | l1_warm_up_steps=5000")
train_toy_sae(sae_25, toy_model, generate_batch)


[info] training sae | d_sae=25 | l1_coefficient=1.0 | l1_warm_up_steps=5000


c:\Users\shpor\miniconda3\envs\cs_449\Lib\site-packages\sae_lens\saes\sae.py:249: UserWarning:


This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)



Training SAE:   0%|          | 0/15000000 [00:00<?, ?it/s]

In [10]:
plot_sae_feat_cos_sims(sae_25, toy_model, "25-latent SAE", dtick=5, reorder_features=True)

In [11]:
sims = cos_sims(sae_25.W_dec.T, toy_model(token_feats).T)
fig = px.imshow(
    sims.detach().cpu().numpy(),
    color_continuous_scale="rdbu",
    zmin=-1,
    zmax=1,
    title="cos sim between SAE decoder and raw token embeddings (25-latent SAE)",
)
fig.show()


It definitely seems like it's learning the token embeddings rather than the underlying features. I'm not sure why it's not getting the token embeddings 100% correct instead of ~85-95% correct, but 🤷